In [44]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [45]:
df = pd.read_csv("fake_news_dataset.csv")
#df.head()

In [46]:
df.columns

Index(['uuid', 'ord_in_thread', 'author', 'published', 'title', 'text',
       'language', 'crawled', 'site_url', 'country', 'domain_rank',
       'thread_title', 'spam_score', 'main_img_url', 'replies_count',
       'participants_count', 'likes', 'comments', 'shares', 'type'],
      dtype='object')

In [47]:
df = df.rename(columns={"content": "text", "target": "label"})

In [48]:
# Keep only needed columns
df = df[['text', 'type']]

# Remove missing text
df = df.dropna(subset=['text'])

# Convert to string
df['text'] = df['text'].astype(str)

# Convert labels to uppercase
df['type'] = df['type'].str.upper()

# Map to binary
df['type'] = df['type'].replace({
    'FAKE': 'FAKE',
    'CONSPIRACY': 'FAKE',
    'HATE': 'FAKE',
    'SATIRE': 'FAKE',
    'JUNKSCI': 'FAKE',
    'BIAS': 'REAL',
    'BS': 'REAL',
    'STATE': 'REAL'
})

# 🔴 IMPORTANT: remove rows where mapping failed
df = df[df['type'].isin(['FAKE', 'REAL'])]

# Reset index (VERY IMPORTANT)
df = df.reset_index(drop=True)

# Labels
fake_labels = ['FAKE', 'CONSPIRACY', 'BIAS', 'HATE', 'SATIRE', 'JUNKSCI']

df['label'] = df['type'].apply(lambda x: 'FAKE' if x in fake_labels else 'REAL')

labels = df['label']

In [49]:
from sklearn.utils import resample

# Combine text + label
df_balanced = pd.concat([df['text'], labels], axis=1)

# Separate classes
df_real = df_balanced[df_balanced['label'] == 'REAL']
df_fake = df_balanced[df_balanced['label'] == 'FAKE']

# Upsample FAKE (increase FAKE samples)
df_fake_upsampled = resample(df_fake,
                            replace=True,
                            n_samples=len(df_real),
                            random_state=42)

# Combine back
df_balanced = pd.concat([df_real, df_fake_upsampled])

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42)

# Final data
X = df_balanced['text']
y = df_balanced['label']

In [50]:
print(df.shape)
print(labels.value_counts())

(12953, 3)
label
REAL    12056
FAKE      897
Name: count, dtype: int64


In [51]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7
)

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)

tfidf_train = tfidf_vectorizer.fit_transform(x_train)
tfidf_test = tfidf_vectorizer.transform(x_test)

In [53]:
from sklearn.linear_model import PassiveAggressiveClassifier

model = PassiveAggressiveClassifier(max_iter=50)
model.fit(tfidf_train, y_train)

PassiveAggressiveClassifier(max_iter=50)

In [54]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(tfidf_test)

score = accuracy_score(y_test, y_pred)
print("Accuracy:", round(score * 100, 2), "%")

Accuracy: 99.07 %


In [55]:
confusion_matrix(y_test, y_pred)

array([[2365,    1],
       [  44, 2413]], dtype=int64)

In [56]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf_vectorizer, open("vectorizer.pkl", "wb"))

In [57]:
for i in range(2):
    print("Text:", x_test.iloc[i][:100])
    print("Actual:", y_test.iloc[i])
    print("Predicted:", y_pred[i])
    print("-"*50)

Text: An international collaboration of astronomers, led by a group at the Swiss Federal Institute of Tech
Actual: REAL
Predicted: REAL
--------------------------------------------------
Text: Google Pinterest Digg Linkedin Reddit Stumbleupon Print Delicious Pocket Tumblr 
By now, we all know
Actual: REAL
Predicted: REAL
--------------------------------------------------
